<a href="https://colab.research.google.com/github/sanmquin/AI/blob/main/src/Graphiko/Fetch-Business-Cluster-Videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fetch videos for business-cluster channels
This notebook reuses the Graphiko business-cluster lookup logic and then fetches Finder videos grouped by channel.


In [1]:
# Install pymongo if not already present
!pip install pymongo dnspython

import pymongo
from pymongo import MongoClient
from google.colab import userdata

try:
    # Retrieve the URI from Colab Secrets
    uri = userdata.get('MONGODB_URI')
    client = MongoClient(uri)

    # Send a ping to confirm connection
    client.admin.command('ping')
    print('✅ Successfully connected to MongoDB!')
except Exception as e:
    print(f'❌ MongoDB connection failed: {e}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 21.3 MB/s eta 0:00:00
✅ Successfully connected to MongoDB!


In [2]:
# Access Finder DB collections and identify the latest business cluster
db = client['finder']
clusters_col = db['ChannelDescriptions_clusters']
items_col = db['ChannelDescriptions_items']
channels_col = db['channels']
videos_col = db['videos']

latest = clusters_col.find_one(sort=[('version', -1), ('createdAt', -1)])
latest_version = latest['version'] if latest else None
print('Latest cluster version:', latest_version)

if latest_version is None:
    print('No clusters found.')
    business_cluster = None
else:
    business_cluster = clusters_col.find_one({
        'version': latest_version,
        'name': {'$regex': '^business', '$options': 'i'}
    })

if business_cluster:
    business_cluster_id = business_cluster['_id']
    print('Business cluster found:', business_cluster.get('name'))
    print('Business cluster _id:', business_cluster_id)
else:
    business_cluster_id = None
    print('No business cluster found in the latest version.')


Latest cluster version: 2
Business cluster found: Business, Venture Capital, and Entrepreneurship
Business cluster _id: 69e41878685f30ed081ec5a6


In [3]:
# Resolve channels associated with the business cluster
if business_cluster_id is None:
    print('Cannot fetch channels without a business cluster id.')
    channel_ids = []
    business_channels = []
else:
    item_docs = list(items_col.find(
        {'clusterId': business_cluster_id},
        {'_id': 0, 'textId': 1}
    ))
    channel_ids = [d['textId'] for d in item_docs]

    business_channels = list(channels_col.find(
        {'channelId': {'$in': channel_ids}},
        {'_id': 0, 'channelId': 1, 'title': 1}
    ))

    print(f'Business cluster has {len(channel_ids)} channel ids and {len(business_channels)} channel docs.')
    for ch in sorted(business_channels, key=lambda x: x.get('title', ''))[:20]:
        print(f"- {ch.get('title', '(untitled)')} ({ch.get('channelId')})")

    if len(business_channels) > 20:
        print(f'... and {len(business_channels) - 20} more channels')


Business cluster has 27 channel ids and 27 channel docs.
- 20VC with Harry Stebbings (UCf0PBRjhf0rF8fWBIxTuoWA)
- ARK Invest (UCK-zlnUfoDHzUwXcbddtnkg)
- Alex Hormozi (UCUyDOdBWhC1MCxEjC46d-zw)
- All-In Podcast (UCESLZhusAkFfsNsApnjF_Cg)
- Anthony Pompliano (UCevXpeL8cNyAnww-NqJ4m2w)
- Asianometry (UC1LpsuAUaKoMzzJSEt5WImw)
- Bg2 Pod (UC-yRDvpR99LUc5l7i7jLzew)
- Bloomberg Originals (UCUMZ7gohGI9HcU9VNsr2FJQ)
- Dan Martell (UCA-mWX9CvCTVFWRMb9bKc9w)
- Garry Tan (UCIBgYfDjtWlbJhg--Z4sOgQ)
- Greg Isenberg (UCPjNBjflYl0-HQtUvOx0Ibw)
- Joe Lonsdale (UCJEDniyP_YtcsXikBELqicw)
- Lenny's Podcast (UC6t1O76G0jYXOAoYCm153dA)
- My First Million (UCyaN6mg5u8Cjy2ZI4ikWaug)
- Network State Podcast (UCKrpnfpTwncQ050VFXcVkuQ)
- Patrick Boyle (UCASM0cgfkJxQ1ICmRilfHLw)
- Peter H. Diamandis (UCvxm0qTrGN_1LMYgUaftWyQ)
- Principles by Ray Dalio (UCqvaXJ1K3HheTPNjH-KpwXQ)
- Real Vision Presents (UCBH5VZE_Y4F3CMcPIzPEB5A)
- Sequoia Capital (UCWrF0oN6unbXrWsTN7RctTw)
... and 7 more channels


## Fetch videos for each business-cluster channel
Retrieves videos from Finder's `videos` collection for all channels in the business cluster and groups them by `channelId`.


In [4]:
from collections import defaultdict

if not channel_ids:
    print('No channel ids available to query videos.')
    videos_by_channel = {}
else:
    cursor = videos_col.find(
        {'channelId': {'$in': channel_ids}},
        {
            '_id': 0,
            'videoId': 1,
            'channelId': 1,
            'title': 1,
            'publishedAt': 1,
            'viewCount': 1
        }
    ).sort([('channelId', 1), ('publishedAt', -1)])

    videos_by_channel = defaultdict(list)
    for doc in cursor:
        videos_by_channel[doc['channelId']].append(doc)

    print(f'Fetched videos for {len(videos_by_channel)} channels.')
    total_videos = sum(len(v) for v in videos_by_channel.values())
    print(f'Total videos fetched: {total_videos}')

    for channel in business_channels:
        cid = channel.get('channelId')
        title = channel.get('title', '(untitled)')
        channel_videos = videos_by_channel.get(cid, [])
        print(f'\n{title} ({cid}) -> {len(channel_videos)} videos')

        # Show up to the 3 most recent videos for quick inspection
        for video in channel_videos[:3]:
            print(f"  • {video.get('title', '(no title)')} | views={video.get('viewCount')} | publishedAt={video.get('publishedAt')}")


Fetched videos for 27 channels.
Total videos fetched: 1344

Bg2 Pod (UC-yRDvpR99LUc5l7i7jLzew) -> 44 videos
  • ChatGPT – The Super Assistant Era | BG2 Guest Interview | views=7574 | publishedAt=2026-03-15 16:25:13
  • AI Enterprise - Databricks & Glean | BG2 Guest Interview | views=24210 | publishedAt=2025-12-23 23:15:04
  • All things AI w @altcap @sama & @satyanadella.  A Halloween Special.  🎃🔥BG2 w/ Brad Gerstner | views=228167 | publishedAt=2025-10-31 23:17:31

The Prof G Pod – Scott Galloway (UC1E1SVcVyU3ntWMSQEp38Yw) -> 50 videos
  • The Hidden Engine of China’s AI Boom | China Decode | views=17697 | publishedAt=2026-04-21 08:00:53
  • Why the SpaceX IPO Doesn't Add Up | Office Hours | views=20314 | publishedAt=2026-04-20 16:01:35
  • S&P and Nasdaq Hit Record Highs as Investors Look Past War | Prof G Markets | views=24154 | publishedAt=2026-04-20 11:05:00

Asianometry (UC1LpsuAUaKoMzzJSEt5WImw) -> 50 videos
  • How To Test 208 Billion Transistors | views=149040 | publishedAt=20

In [5]:
# Optional: flatten all fetched videos to a DataFrame for downstream analysis/export
import pandas as pd

all_videos = [v for vids in videos_by_channel.values() for v in vids] if videos_by_channel else []
videos_df = pd.DataFrame(all_videos)
print('videos_df shape:', videos_df.shape)
videos_df.head(10)


videos_df shape: (1344, 5)


,videoId,channelId,publishedAt,title,viewCount
0,MIKej1HCRW0,UC-yRDvpR99LUc5l7i7jLzew,2026-03-15 16:25:13,ChatGPT – The Super Assistant Era | BG2 Guest ...,7574
1,jA8ZQfq_Hzs,UC-yRDvpR99LUc5l7i7jLzew,2025-12-23 23:15:04,AI Enterprise - Databricks & Glean | BG2 Guest...,24210
2,Gnl833wXRz0,UC-yRDvpR99LUc5l7i7jLzew,2025-10-31 23:17:31,All things AI w @altcap @sama & @satyanadella....,228167
3,KX6q6lvoYtM,UC-yRDvpR99LUc5l7i7jLzew,2025-10-14 23:56:17,"AI Bubble, Stablecoin Boom, and Runnin' Down a...",71463
4,pE6sw_E9Gh0,UC-yRDvpR99LUc5l7i7jLzew,2025-09-26 03:36:04,"NVIDIA: OpenAI, Future of Compute, and the Ame...",426866
5,yLTSqBzKG2s,UC-yRDvpR99LUc5l7i7jLzew,2025-09-11 21:25:36,Inside OpenAI Enterprise: Forward Deployed Eng...,23451
6,hUJz55AsUz4,UC-yRDvpR99LUc5l7i7jLzew,2025-08-28 23:51:07,"China, China, China. Breaking Down China’s Tec...",114510
7,fTqINzeudJ4,UC-yRDvpR99LUc5l7i7jLzew,2025-07-31 15:25:00,"China Open-Source, Compute Arms Race, Reorderi...",54888
8,X52BNWZrXSk,UC-yRDvpR99LUc5l7i7jLzew,2025-07-10 23:23:42,"Michael Dell – Invest America Act Becomes Law,...",75922
9,IOoRXSyezBE,UC-yRDvpR99LUc5l7i7jLzew,2025-06-21 16:36:30,Coatue Pt2. Open AI’s Kevin Weil Dives into Al...,42867
